# E3 — 02 — SVM

Experimentos con el SVM, para las dos tareas (detección y tipo):
- **E1** (secciones 5 y 6): ¿qué representación funciona mejor, el TF-IDF o el vector medio de eswiki (+ nº de `?` y `!`)?
- **E3** (sección 7): con la mejor representación, ¿es mejor entrenar con los tres corpus o solo con Political?

- En el E1 se entrena con el train de los tres corpus mezclados.
- Los hiperparámetros se eligen con Optuna, mirando solo el **dev de Political**. El test no se mira aquí: las métricas finales salen de `06_comparativa`.
- Cada combinación de tarea y representación tiene su propia búsqueda, así se compara el mejor SVM de cada representación.

| Salida | Contenido |
|---|---|
| `predicciones/{tarea}__svm__{variante}__s0.csv` | Predicciones de dev y test (`id`, `split`, `real`, `predicho`) |
| `mejores_params_svm.json` | Mejores hiperparámetros de cada configuración, para regenerar los CSV sin repetir Optuna |

El SVM se entrena una sola vez (`s0`): su entrenamiento es convexo (tiene un único mínimo), así que con otra semilla sale el mismo modelo.

## 1. Datos

Todo viene de `data/e3/` (notebook `01_datos`). La fila *i* de `datos_E3.jsonl` es la fila *i* de `tfidf.npz` y de `features_extra.npy`, así que el índice de la tabla sirve para sacar las filas de las matrices.

In [1]:
import json
import os

import numpy as np
import optuna
import pandas as pd
from scipy import sparse
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

CARPETA = "../data/e3"

df = pd.read_json(f"{CARPETA}/datos_E3.jsonl", lines=True)
tfidf = sparse.load_npz(f"{CARPETA}/tfidf.npz")
extra = np.load(f"{CARPETA}/features_extra.npy")

print(len(df), tfidf.shape, extra.shape)

C:\Users\asier.gomez\Proyectos\Clase\detector_falacias_contradicciones\nlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


7843 (7843, 4503) (7843, 302)


## 2. Las dos representaciones

- `tfidf`: tal cual. Cada fila ya tiene tamaño 1 (normalización L2 del TF-IDF).
- `embedding`: vector medio de eswiki (300 columnas) + nº de `?` y `!` (2 columnas). Necesita dos ajustes:
    1. `StandardScaler`, ajustado solo con train: deja cada columna con media 0 y desviación 1. Sin él, los recuentos de `?` y `!` (hasta 9) pesarían más que las columnas del vector medio (desviación 0,05) solo por su escala.
    2. Dividir por √302: cada texto pasa a tener tamaño 1 de media, como en el TF-IDF. No cambia los modelos que se pueden aprender (equivale a cambiar `C`), pero así el mismo rango de `C` significa lo mismo en las dos representaciones. Sin esto, el SVM lineal con el embedding tarda minutos en cada intento.

In [2]:
escalador = StandardScaler().fit(extra[df["split"] == "train"])
embedding = escalador.transform(extra) / np.sqrt(extra.shape[1])

REPRESENTACIONES = {"tfidf": tfidf, "embedding": embedding}

## 3. Las dos tareas

- **Detección:** todos los textos, con `etiqueta_deteccion` (0 sin falacia, 1 falacia).
- **Tipo:** solo los textos con falacia, con `etiqueta_tipo`. Es una lista: `explode` deja una fila por etiqueta. En train duplica los 9 textos con dos etiquetas; en dev y test todos tienen una sola, así que solo la saca de la lista.

`separar` devuelve el train (los tres corpus) y el dev de Political, que es con lo que se eligen los hiperparámetros.

In [3]:
deteccion = df.assign(etiqueta=df["etiqueta_deteccion"])
tipo = df[df["etiqueta_tipo"].notna()].explode("etiqueta_tipo")
tipo = tipo.assign(etiqueta=tipo["etiqueta_tipo"])

TAREAS = {"deteccion": deteccion, "tipo": tipo}


def separar(tabla, X):
    train = tabla[tabla["split"] == "train"]
    dev = tabla[(tabla["split"] == "dev") & (tabla["corpus"] == "political")]
    return X[train.index], train["etiqueta"], X[dev.index], dev["etiqueta"]


for nombre, tabla in TAREAS.items():
    X_tr, y_tr, X_dv, y_dv = separar(tabla, tfidf)
    print(nombre, "train:", len(y_tr), " dev Political:", len(y_dv))

deteccion train: 5512  dev Political: 401
tipo train: 3200  dev Political: 289


## 4. Funciones comunes

- `crear_svm`: el SVM, con `class_weight="balanced"` para el desbalance (cada clase pesa más cuanto más rara es).
- `buscar`: la búsqueda de Optuna. En cada intento entrena con train y lo puntúa con el F1 macro del dev de Political. Busca:
    - `kernel`: lineal (frontera recta) o RBF (frontera curva).
    - `C`, entre 0,001 y 100 en escala logarítmica: el precio de cada error en train. Pequeño, margen ancho y modelo simple; grande, intenta acertar todo train y puede sobreajustar.
    - `gamma`, solo con RBF, entre 0,0001 y 10: lo lejos que llega la influencia de cada ejemplo. Grande, frontera muy ajustada a cada punto.

  El sampler TPE (Bergstra et al., 2011) hace 10 intentos al azar y después concentra los siguientes donde los anteriores dieron mejor F1. Con `seed=42` la búsqueda sale igual cada vez.
- `guardar_predicciones` y `guardar_params`: los CSV de predicciones y el JSON de hiperparámetros.
- `experimento`: los pasos de una configuración, en orden.

In [4]:
N_INTENTOS = 40
RUTA_PARAMS = "mejores_params_svm.json"
optuna.logging.set_verbosity(optuna.logging.WARNING)


def crear_svm(params):
    return SVC(class_weight="balanced", **params)


def f1_dev(modelo, X_dev, y_dev):
    return f1_score(y_dev, modelo.predict(X_dev), average="macro", zero_division=0)


def buscar(X_train, y_train, X_dev, y_dev):
    def intento(trial):
        params = {
            "kernel": trial.suggest_categorical("kernel", ["linear", "rbf"]),
            "C": trial.suggest_float("C", 1e-3, 100, log=True),
        }
        if params["kernel"] == "rbf":
            params["gamma"] = trial.suggest_float("gamma", 1e-4, 10, log=True)
        modelo = crear_svm(params).fit(X_train, y_train)
        return f1_dev(modelo, X_dev, y_dev)

    estudio = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
    estudio.optimize(intento, n_trials=N_INTENTOS)
    return estudio.best_params


def guardar_predicciones(modelo, tabla, X, tarea, variante):
    filas = tabla[tabla["split"].isin(["dev", "test"])]
    salida = pd.DataFrame({
        "id": filas["id"],
        "split": filas["split"],
        "real": filas["etiqueta"],
        "predicho": modelo.predict(X[filas.index]),
    })
    os.makedirs("predicciones", exist_ok=True)
    salida.to_csv(f"predicciones/{tarea}__svm__{variante}__s0.csv", index=False)


def guardar_params(clave, params):
    # se lee y se reescribe, para no borrar las otras configuraciones si se ejecuta una sola
    todos = {}
    if os.path.exists(RUTA_PARAMS):
        with open(RUTA_PARAMS) as f:
            todos = json.load(f)
    todos[clave] = params
    with open(RUTA_PARAMS, "w") as f:
        json.dump(todos, f, indent=2)


def experimento(tarea, variante):
    tabla, X = TAREAS[tarea], REPRESENTACIONES[variante]
    X_tr, y_tr, X_dv, y_dv = separar(tabla, X)

    params = buscar(X_tr, y_tr, X_dv, y_dv)                  # 1. Optuna, con el dev de Political
    modelo = crear_svm(params).fit(X_tr, y_tr)               # 2. modelo final, solo con train
    guardar_predicciones(modelo, tabla, X, tarea, variante)  # 3. CSV de dev y test
    guardar_params(f"{tarea}__{variante}", params)           # 4. JSON

    return {"tarea": tarea, "variante": variante, **params,
            "F1 macro dev Political": round(f1_dev(modelo, X_dv, y_dv), 3)}

## 5.1. Detección con TF-IDF

In [5]:
r1 = experimento("deteccion", "tfidf")
r1

{'tarea': 'deteccion',
 'variante': 'tfidf',
 'kernel': 'rbf',
 'C': 6.509924253235056,
 'gamma': 0.08971620093533346,
 'F1 macro dev Political': 0.724}

## 5.2. Detección con embedding

In [6]:
r2 = experimento("deteccion", "embedding")
r2

{'tarea': 'deteccion',
 'variante': 'embedding',
 'kernel': 'rbf',
 'C': 2.7430913692110273,
 'gamma': 0.7314476796915378,
 'F1 macro dev Political': 0.74}

## 5.3. Tipo con TF-IDF

In [7]:
r3 = experimento("tipo", "tfidf")
r3

{'tarea': 'tipo',
 'variante': 'tfidf',
 'kernel': 'linear',
 'C': 0.5061643778066425,
 'F1 macro dev Political': 0.304}

## 5.4. Tipo con embedding

In [8]:
r4 = experimento("tipo", "embedding")
r4

{'tarea': 'tipo',
 'variante': 'embedding',
 'kernel': 'linear',
 'C': 2.768418684383134,
 'F1 macro dev Political': 0.342}

## 6. Resumen

F1 macro en el dev de Political de cada configuración, con sus hiperparámetros. Sirve para elegir la mejor representación del SVM en cada tarea; la comparación con los otros modelos y las métricas del test se hacen en `06_comparativa`.

In [9]:
pd.DataFrame([r1, r2, r3, r4])

,tarea,variante,kernel,C,gamma,F1 macro dev Political
0,deteccion,tfidf,rbf,6.509924,0.089716,0.724
1,deteccion,embedding,rbf,2.743091,0.731448,0.740
2,tipo,tfidf,linear,0.506164,NaN,0.304
3,tipo,embedding,linear,2.768419,NaN,0.342


## 7. E3: solo Political frente a mezclado

El SVM es el mejor modelo clásico en las dos tareas, y su mejor representación en ambas es el `embedding` (sección 6). Aquí se entrena otra vez, pero **solo con el train de Political**:
- Detección: 1.870 textos en vez de 5.512.
- Tipo: 1.325 en vez de 3.200.

Se usan los mismos hiperparámetros que con el train mezclado (del JSON), sin repetir Optuna: así lo único que cambia son los datos de entrenamiento. Dev y test no cambian.

Responde a si el lenguaje informal de los otros dos corpus ayuda a Political o lo lía. Las predicciones se guardan como `{tarea}__svm__embedding_solo_political__s0.csv`.

In [10]:
def experimento_solo_political(tarea, variante):
    tabla, X = TAREAS[tarea], REPRESENTACIONES[variante]
    with open(RUTA_PARAMS) as f:
        params = json.load(f)[f"{tarea}__{variante}"]

    solo_political = tabla[(tabla["split"] != "train") | (tabla["corpus"] == "political")]
    X_tr, y_tr, X_dv, y_dv = separar(solo_political, X)
    modelo = crear_svm(params).fit(X_tr, y_tr)
    guardar_predicciones(modelo, tabla, X, tarea, f"{variante}_solo_political")

    return {"tarea": tarea, "variante": f"{variante}_solo_political", **params,
            "F1 macro dev Political": round(f1_dev(modelo, X_dv, y_dv), 3)}

F1 macro en el dev de Political, mezclado (de la sección 5) frente a solo Political:

In [11]:
e3_deteccion = experimento_solo_political("deteccion", "embedding")
e3_tipo = experimento_solo_political("tipo", "embedding")

pd.DataFrame([r2, e3_deteccion, r4, e3_tipo])

,tarea,variante,kernel,C,gamma,F1 macro dev Political
0,deteccion,embedding,rbf,2.743091,0.731448,0.740
1,deteccion,embedding_solo_political,rbf,2.743091,0.731448,0.704
2,tipo,embedding,linear,2.768419,NaN,0.342
3,tipo,embedding_solo_political,linear,2.768419,NaN,0.226
